# 14 — Selección de variables y features de trayectoria

**Objetivo general del cuaderno:** partir de `df_unificado.csv` (panel largo país-año-indicador,
2000-2020, producido en el cuaderno 13) y construir:

1. El **target de inflación** aislado y excluido del set de variables de clustering (`FP.CPI.TOTL.ZG`,
   con `NY.GDP.DEFL.KD.ZG` como target secundario de robustez).
2. Una **tabla de disponibilidad** por variable candidata restante (cobertura de países y de años),
   para aplicar el criterio de selección con evidencia, no por intuición.
3. La **matriz país × variable** con tres estadísticos de trayectoria por variable (media, tendencia,
   volatilidad residual), lista para el PCA por dimensión (cuaderno 15).

**Por qué "trayectoria resumida" y no clustering de series temporales real (DTW, etc.):** decisión
documentada por restricción de alcance/tiempo — ver párrafo de limitaciones ya redactado para el TFM.


## 1. Configuración e importaciones

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

pd.set_option("display.max_columns", None)


## 2. Carga de datos

**Objetivo:** cargar `df_unificado.csv` (salida del cuaderno 13).

**Justificación metodológica:** es el único insumo necesario para este cuaderno; ya contiene la
unión WDI + V-Dem + MC con `fuente` y `dimension_principal` anexadas.

**Resultado esperado:** `df` con ~4,2M filas y columnas
`ISO-alpha3, Country or Area, M49_region, Region Name, M49_subregion, Sub-region Name, year, codigo,
valor, fuente, dimension_principal`.


In [2]:
IN_PATH = Path("data/processed/df_unificado.csv")
df = pd.read_csv(IN_PATH)

print(f"Shape df_unificado: {df.shape}")
print(f"Rango de anios: {df['year'].min()} - {df['year'].max()}")
print(f"Paises unicos: {df['ISO-alpha3'].nunique()}")
print(f"Codigos unicos: {df['codigo'].nunique()}")
df.head()


Shape df_unificado: (4239527, 11)
Rango de anios: 2000 - 2020
Paises unicos: 193
Codigos unicos: 1055


,ISO-alpha3,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,year,codigo,valor,fuente,dimension_principal
0,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.ZS,4.4,WDI,SOCIODEMOGRAFICA
1,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.RU.ZS,NaN,WDI,SOCIODEMOGRAFICA
2,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,EG.ELC.ACCS.UR.ZS,73.4,WDI,SOCIODEMOGRAFICA
3,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,FX.OWN.TOTL.ZS,NaN,WDI,SOCIODEMOGRAFICA
4,AFG,Afghanistan,142,Asia,34,Southern Asia,2000,FX.OWN.TOTL.FE.ZS,NaN,WDI,SOCIODEMOGRAFICA


**Qué comprobar:**
- Shape debería ser cercano a (4.239.527, 11), tal como quedó documentado en el cuaderno 13.
- Si `year` no llega a 2000-2020 completo, revisar que el CSV cargado sea el correcto (no una
  versión vieja en `data/processed/`).


## 3. Aislar el target de inflación

**Objetivo:** separar las variables de inflación del resto de variables candidatas **antes** de
cualquier análisis de disponibilidad o de trayectoria, para que no puedan colarse en el set de
clustering de la dimensión MONETARIA.

**Justificación metodológica:** la inflación es la variable de resultado que se busca caracterizar
mediante los perfiles (etapa 1: clustering macro-fiscal *excluyendo* inflación; etapa 2: análisis de
qué configuraciones institucionales/estructurales/sociodemográficas se asocian a trayectorias
divergentes de inflación *dentro* de cada grupo macro-fiscal). Si la inflación entra al clustering de
etapa 1, los grupos quedarían definidos parcialmente por la propia variable que se quiere diferenciar
entre grupos — un error circular.

Se excluyen del set de clustering **las cuatro variables de inflación/precios** presentes en el
catálogo (`FP.CPI.TOTL`, `FP.CPI.TOTL.ZG`, `NY.GDP.DEFL.KD.ZG`, `NY.GDP.DEFL.KD.ZG.AD`), no solo el
target principal — de lo contrario alguna quedaría en MONETARIA como proxy casi perfecto de la propia
inflación. `FP.CPI.TOTL.ZG` (inflación IPC, % anual) queda como target principal; `NY.GDP.DEFL.KD.ZG`
(deflactor del PIB) como target secundario de robustez, si da tiempo de usarlo.

**Resultado esperado:** `df_target_inflacion` (target principal, país-año), `df_target_secundario`
(deflactor, país-año), `df_variables` (resto de candidatas, sin ninguna de las 4 variables de
precios/inflación).


In [3]:
CODIGO_TARGET_PRINCIPAL = "FP.CPI.TOTL.ZG"
CODIGO_TARGET_SECUNDARIO = "NY.GDP.DEFL.KD.ZG"

CODIGOS_PRECIOS_EXCLUIR = [
    "FP.CPI.TOTL",           # nivel del indice, no comparable entre paises (base 2010=100)
    "FP.CPI.TOTL.ZG",        # target principal
    "NY.GDP.DEFL.KD.ZG",     # target secundario (robustez)
    "NY.GDP.DEFL.KD.ZG.AD",  # serie enlazada del deflactor, redundante con la anterior
]

df_target_inflacion = df[df["codigo"] == CODIGO_TARGET_PRINCIPAL].copy()
df_target_secundario = df[df["codigo"] == CODIGO_TARGET_SECUNDARIO].copy()
df_variables = df[~df["codigo"].isin(CODIGOS_PRECIOS_EXCLUIR)].copy()

print(f"Filas target principal ({CODIGO_TARGET_PRINCIPAL}): {df_target_inflacion.shape[0]}")
print(f"Paises con al menos un dato de inflacion (target principal): {df_target_inflacion['ISO-alpha3'].nunique()}")
print(f"Filas target secundario ({CODIGO_TARGET_SECUNDARIO}): {df_target_secundario.shape[0]}")
print(f"Filas resto de variables (candidatas a clustering): {df_variables.shape[0]}")
print(f"Codigos candidatos restantes: {df_variables['codigo'].nunique()}")

assert not set(CODIGOS_PRECIOS_EXCLUIR) & set(df_variables["codigo"].unique()), \
    "Alguna variable de precios/inflacion sigue en df_variables"


Filas target principal (FP.CPI.TOTL.ZG): 4053
Paises con al menos un dato de inflacion (target principal): 193
Filas target secundario (NY.GDP.DEFL.KD.ZG): 4053
Filas resto de variables (candidatas a clustering): 4223315
Codigos candidatos restantes: 1051


**Qué comprobar:**
- `df_target_inflacion['ISO-alpha3'].nunique()` idealmente cerca de 193; si es sensiblemente menor,
  documentarlo — afecta directamente cuántos países entran al análisis final.
- El assert final debe pasar sin error.


## 4. Disponibilidad por variable candidata

**Objetivo:** para cada código candidato restante, calcular cobertura de países (cuántos de los 193
tienen al menos un dato) y cobertura temporal promedio (cuántos de los 21 años, en promedio, entre
los países que sí tienen algún dato).

**Justificación metodológica:** principio de mínima intervención y de nunca eliminar variables por
intuición — se necesita evidencia empírica de cobertura antes de decidir qué entra al análisis final.
La cobertura temporal importa especialmente acá porque la tendencia y la volatilidad residual
requieren un mínimo de puntos para ser confiables (ver sección 6).

**Resultado esperado:** `df_disponibilidad_variables`, con `codigo, fuente, dimension_principal,
n_paises_con_dato, pct_paises, pct_anios_promedio`.


In [4]:
N_PAISES_UNIVERSO = 193
N_ANIOS_UNIVERSO = 21  # 2000-2020 inclusive

df_variables_validas = df_variables.dropna(subset=["valor"])

resumen_cobertura = (
    df_variables_validas
    .groupby("codigo")
    .agg(
        fuente=("fuente", "first"),
        dimension_principal=("dimension_principal", "first"),
        n_paises_con_dato=("ISO-alpha3", "nunique"),
        n_obs=("valor", "count"),
    )
    .reset_index()
)
resumen_cobertura["pct_paises"] = (resumen_cobertura["n_paises_con_dato"] / N_PAISES_UNIVERSO * 100).round(1)

cobertura_temporal = (
    df_variables_validas
    .groupby(["codigo", "ISO-alpha3"])["year"]
    .nunique()
    .reset_index(name="n_anios_con_dato")
)
cobertura_temporal_prom = (
    cobertura_temporal.groupby("codigo")["n_anios_con_dato"]
    .mean()
    .reset_index(name="prom_anios_con_dato_pais")
)
cobertura_temporal_prom["pct_anios_promedio"] = (
    cobertura_temporal_prom["prom_anios_con_dato_pais"] / N_ANIOS_UNIVERSO * 100
).round(1)

df_disponibilidad_variables = resumen_cobertura.merge(
    cobertura_temporal_prom[["codigo", "pct_anios_promedio"]], on="codigo", how="left"
)

# Codigos candidatos que no tienen NI UN dato valido en todo el panel (documentar, no perder)
codigos_sin_ningun_dato = sorted(set(df_variables["codigo"].unique()) - set(df_disponibilidad_variables["codigo"]))

print(f"Shape df_disponibilidad_variables: {df_disponibilidad_variables.shape}")
print(f"Codigos candidatos sin NINGUN dato valido: {len(codigos_sin_ningun_dato)}")
if codigos_sin_ningun_dato:
    print(codigos_sin_ningun_dato)

df_disponibilidad_variables.sort_values("pct_paises").head(15)


Shape df_disponibilidad_variables: (1041, 7)
Codigos candidatos sin NINGUN dato valido: 10
['DT.DOD.PVLX.CD', 'DT.DOD.PVLX.EX.ZS', 'DT.DOD.PVLX.GN.ZS', 'GD_WBL_OVL_LAW', 'GD_WBL_OVL_SFR', 'IC.BRE.MC.OS', 'IC.BRE.MC.P1', 'IC.BRE.MC.P2', 'IC.BRE.MC.P3', 'SM.POP.RRWA.EO']


,codigo,fuente,dimension_principal,n_paises_con_dato,n_obs,pct_paises,pct_anios_promedio
765,SM.POP.OPIP.EO,WDI,SOCIODEMOGRAFICA,1,3,0.5,14.3
768,SM.POP.RRWA.EA,WDI,SOCIODEMOGRAFICA,3,63,1.6,100.0
764,SM.POP.OPIP.EA,WDI,SOCIODEMOGRAFICA,20,51,10.4,12.1
118,FP.WPI.TOTL,WDI,MONETARIA,29,423,15.0,69.5
570,SH.STA.FGMS.ZS,WDI,SOCIODEMOGRAFICA,30,73,15.5,11.6
938,per_lm_alllm.adq_pop_tot,WDI,MONETARIA,32,103,16.6,15.3
939,per_lm_alllm.ben_q1_tot,WDI,ESTRUCTURAL,32,103,16.6,15.3
644,SI.RMT.COST.OB.ZS,WDI,MONETARIA,46,220,23.8,22.8
125,FS.AST.DOMO.GD.ZS,WDI,MONETARIA,59,747,30.6,60.3
645,SL.AGR.0714.FE.ZS,WDI,ESTRUCTURAL,62,141,32.1,10.8


### Distribución de cobertura

Objetivo: visualizar cómo se distribuyen pct_paises y pct_anios_promedio entre las ~1.051 variables candidatas, antes de fijar ningún umbral.

Justificación metodológica: un histograma permite ver si existe un "codo" natural (un grupo claramente bien cubierto separado de uno mal cubierto) o si la cobertura es un continuo sin quiebres.

Resultado esperado: dos histogramas (uno por métrica), con líneas verticales marcando el umbral de partida (70% / 60%) como referencia visual, no como resultado ya decidido.

In [5]:
import plotly.express as px
import plotly.graph_objects as go

fig_hist_paises = px.histogram(
    df_disponibilidad_variables, x="pct_paises", nbins=40,
    title="Distribución de cobertura de países por variable candidata",
    labels={"pct_paises": "% de países con al menos un dato"}
)
fig_hist_paises.add_vline(x=70, line_dash="dash", line_color="red",
                           annotation_text="umbral propuesto 70%")
fig_hist_paises.show()

fig_hist_anios = px.histogram(
    df_disponibilidad_variables, x="pct_anios_promedio", nbins=40,
    title="Distribución de cobertura temporal promedio por variable candidata",
    labels={"pct_anios_promedio": "% de años (promedio) con dato, entre países con algún dato"}
)
fig_hist_anios.add_vline(x=60, line_dash="dash", line_color="red",
                          annotation_text="umbral propuesto 60%")
fig_hist_anios.show()

### Hostograma 1:

cada barra indica cuántos indicadores (no cuántos países) caen dentro de un rango determinado de cobertura, de modo que una barra alta cerca del 100% significa que hay muchos indicadores con cobertura casi completa, y una barra baja cerca del 0% significa que pocos indicadores tienen cobertura tan pobre.

### Histograma 2:

Para los países que sí tienen algún dato de ese indicador, ¿qué porcentaje de los 21 años (en promedio) están efectivamente cubiertos?


### Curva de sensibilidad del umbral
Objetivo: para un rango de umbrales candidatos (no solo 70/60), calcular cuántas variables sobrevivirían bajo cada uno.

Justificación metodológica: muestra si 70/60 cae en una zona "plana" de la curva (poco sensible al valor exacto elegido — buena señal de robustez) o en una pendiente pronunciada (pequeños cambios en el umbral alteran mucho el resultado — señal de que el número exacto importa y merece más justificación).

Resultado esperado: gráfico de línea con el eje x = umbral de pct_paises (manteniendo pct_anios_promedio fijo en 60, y viceversa en un segundo panel), eje y = número de variables retenidas.

In [6]:
umbrales_paises = list(range(40, 96, 5))
umbrales_anios = list(range(30, 91, 5))

retenidas_por_umbral_paises = [
    ((df_disponibilidad_variables["pct_paises"] >= u)
     & (df_disponibilidad_variables["pct_anios_promedio"] >= 60)).sum()
    for u in umbrales_paises
]

retenidas_por_umbral_anios = [
    ((df_disponibilidad_variables["pct_paises"] >= 70)
     & (df_disponibilidad_variables["pct_anios_promedio"] >= u)).sum()
    for u in umbrales_anios
]

fig_sens = go.Figure()
fig_sens.add_trace(go.Scatter(x=umbrales_paises, y=retenidas_por_umbral_paises,
                               mode="lines+markers", name="Variando umbral % países (años fijo=60)"))
fig_sens.add_trace(go.Scatter(x=umbrales_anios, y=retenidas_por_umbral_anios,
                               mode="lines+markers", name="Variando umbral % años (países fijo=70)"))
fig_sens.add_vline(x=70, line_dash="dash", line_color="gray")
fig_sens.update_layout(title="Sensibilidad: variables retenidas según umbral",
                        xaxis_title="Umbral (%)", yaxis_title="N variables retenidas")
fig_sens.show()

### Cobertura por dimensión analítica

Objetivo: ver si el problema de cobertura es homogéneo entre las cuatro dimensiones teóricas o si alguna (particularmente ESTRUCTURAL, que depende de una única fuente MC) está sistemáticamente peor.

Justificación metodológica: un umbral único global asume implícitamente que las 4 dimensiones tienen calidad de datos comparable. Si no es así, aplicar el mismo corte a todas puede vaciar desproporcionadamente una dimensión — mejor detectarlo acá, visualmente, que en el PCA del cuaderno 15.

Resultado esperado: scatter pct_paises vs pct_anios_promedio, coloreado por dimension_principal, con líneas de referencia del umbral propuesto.

In [7]:
fig_scatter = px.scatter(
    df_disponibilidad_variables, x="pct_paises", y="pct_anios_promedio",
    color="dimension_principal", hover_data=["codigo", "fuente"],
    title="Cobertura de países vs. cobertura temporal, por dimensión analítica",
    opacity=0.6
)
fig_scatter.add_vline(x=70, line_dash="dash", line_color="gray")
fig_scatter.add_hline(y=60, line_dash="dash", line_color="gray")
fig_scatter.show()

In [8]:
umbrales_conjunto = list(range(0, 101, 3))
dimensiones = df_disponibilidad_variables["dimension_principal"].unique()

registros_sensibilidad_dim = []
for t in umbrales_conjunto:
    cumple_t = (
        (df_disponibilidad_variables["pct_paises"] >= t)
        & (df_disponibilidad_variables["pct_anios_promedio"] >= t)
    )
    for dim in dimensiones:
        mask_dim = df_disponibilidad_variables["dimension_principal"] == dim
        n_total_dim = mask_dim.sum()
        n_retenidas_dim = (mask_dim & cumple_t).sum()
        pct_retenido = (n_retenidas_dim / n_total_dim * 100) if n_total_dim > 0 else np.nan
        registros_sensibilidad_dim.append({
            "umbral": t, "dimension_principal": dim,
            "pct_retenido": pct_retenido, "n_total_dim": n_total_dim,
        })

df_sensibilidad_dim = pd.DataFrame(registros_sensibilidad_dim)

fig_sens_dim = px.line(
    df_sensibilidad_dim, x="umbral", y="pct_retenido", color="dimension_principal",
    markers=True,
    title="Sensibilidad del umbral conjunto por dimensión analítica (% de variables retenidas)",
    labels={"umbral": "Umbral aplicado (% países Y % años)", "pct_retenido": "% de variables retenidas (dentro de la dimensión)"}
)
fig_sens_dim.show()

In [9]:
UMBRAL_PCT_PAISES = 70
UMBRAL_PCT_ANIOS = 60

cumple_umbral = (
    (df_disponibilidad_variables["pct_paises"] >= UMBRAL_PCT_PAISES)
    & (df_disponibilidad_variables["pct_anios_promedio"] >= UMBRAL_PCT_ANIOS)
)

resumen_umbral_dim = df_disponibilidad_variables.groupby("dimension_principal").agg(
    n_total=("codigo", "count"),
    n_retenidas=("codigo", lambda s: cumple_umbral[s.index].sum()),
)
resumen_umbral_dim["pct_retenido"] = (resumen_umbral_dim["n_retenidas"] / resumen_umbral_dim["n_total"] * 100).round(1)
resumen_umbral_dim = resumen_umbral_dim.sort_values("n_retenidas")

resumen_umbral_dim

,n_total,n_retenidas,pct_retenido
dimension_principal,,,
INSTITUCIONAL,122,80,65.6
MONETARIA,191,141,73.8
ESTRUCTURAL,241,207,85.9
SOCIODEMOGRAFICA,487,258,53.0


## 5. Umbral de disponibilidad y filtro de variables

**Objetivo:** aplicar umbrales mínimos de cobertura (países y años) para decidir qué variables entran
al análisis final, documentando explícitamente las excluidas y el motivo.

**Justificación metodológica:** dos umbrales, con lógica distinta:
- **`pct_paises`** — asegura comparabilidad entre países: una variable con muy pocos países no aporta
  a un clustering que busca comparar 193 (o menos) unidades.
- **`pct_anios_promedio`** — asegura que la tendencia y la volatilidad (sección 6) se calculen con
  suficientes puntos como para ser confiables, no solo con 2-3 años.

Umbrales de partida (ajustables): **≥70% de países** (≈135/193) y **≥60% de años promedio** (≈13/21).
Son un punto de partida razonable dado el poco tiempo disponible — más estricto perdería demasiadas
variables; más laxo comprometería la calidad de las tendencias. Si el resultado final deja muy pocas
variables en alguna dimensión (p. ej. ESTRUCTURAL, que ya depende de una sola fuente MC), conviene
revisar el umbral solo para esa dimensión antes de seguir.

**Resultado esperado:** `codigos_incluidos`, `df_variables_excluidas` (documentadas con motivo).


In [10]:
UMBRAL_PCT_PAISES = 70
UMBRAL_PCT_ANIOS = 60

df_disponibilidad_variables["cumple_umbral"] = (
    (df_disponibilidad_variables["pct_paises"] >= UMBRAL_PCT_PAISES)
    & (df_disponibilidad_variables["pct_anios_promedio"] >= UMBRAL_PCT_ANIOS)
)

codigos_incluidos = df_disponibilidad_variables.loc[
    df_disponibilidad_variables["cumple_umbral"], "codigo"
].tolist()

print(f"Variables que cumplen umbral ({UMBRAL_PCT_PAISES}% paises, {UMBRAL_PCT_ANIOS}% anios): {len(codigos_incluidos)}")
print("\nDistribucion por dimension (incluidas):")
print(
    df_disponibilidad_variables.loc[df_disponibilidad_variables["cumple_umbral"], "dimension_principal"]
    .value_counts()
)


Variables que cumplen umbral (70% paises, 60% anios): 686

Distribucion por dimension (incluidas):
dimension_principal
SOCIODEMOGRAFICA    258
ESTRUCTURAL         207
MONETARIA           141
INSTITUCIONAL        80
Name: count, dtype: int64


In [11]:
# Documentacion de variables excluidas (por umbral o por ausencia total de dato) - nunca se pierden en silencio
df_excluidas_por_umbral = df_disponibilidad_variables.loc[
    ~df_disponibilidad_variables["cumple_umbral"],
    ["codigo", "fuente", "dimension_principal", "pct_paises", "pct_anios_promedio"],
].copy()
df_excluidas_por_umbral["motivo_exclusion"] = "baja_cobertura_umbral"

df_metadata_candidatos = df.drop_duplicates("codigo")[["codigo", "fuente", "dimension_principal"]]
df_excluidas_sin_dato = df_metadata_candidatos[
    df_metadata_candidatos["codigo"].isin(codigos_sin_ningun_dato)
].copy()
df_excluidas_sin_dato["pct_paises"] = 0.0
df_excluidas_sin_dato["pct_anios_promedio"] = 0.0
df_excluidas_sin_dato["motivo_exclusion"] = "sin_ningun_dato_valido"

df_variables_excluidas = pd.concat([df_excluidas_por_umbral, df_excluidas_sin_dato], ignore_index=True)

print(f"Total variables excluidas (documentadas): {df_variables_excluidas.shape[0]}")
print(df_variables_excluidas["motivo_exclusion"].value_counts())
df_variables_excluidas.sort_values("pct_paises").head(10)


Total variables excluidas (documentadas): 365
motivo_exclusion
baja_cobertura_umbral     355
sin_ningun_dato_valido     10
Name: count, dtype: int64


,codigo,fuente,dimension_principal,pct_paises,pct_anios_promedio,motivo_exclusion
364,GD_WBL_OVL_SFR,WDI,INSTITUCIONAL,0.0,0.0,sin_ningun_dato_valido
355,IC.BRE.MC.P1,WDI,ESTRUCTURAL,0.0,0.0,sin_ningun_dato_valido
356,IC.BRE.MC.P2,WDI,ESTRUCTURAL,0.0,0.0,sin_ningun_dato_valido
357,IC.BRE.MC.P3,WDI,ESTRUCTURAL,0.0,0.0,sin_ningun_dato_valido
358,IC.BRE.MC.OS,WDI,ESTRUCTURAL,0.0,0.0,sin_ningun_dato_valido
363,GD_WBL_OVL_LAW,WDI,INSTITUCIONAL,0.0,0.0,sin_ningun_dato_valido
360,DT.DOD.PVLX.GN.ZS,WDI,MONETARIA,0.0,0.0,sin_ningun_dato_valido
361,DT.DOD.PVLX.CD,WDI,MONETARIA,0.0,0.0,sin_ningun_dato_valido
362,SM.POP.RRWA.EO,WDI,SOCIODEMOGRAFICA,0.0,0.0,sin_ningun_dato_valido
359,DT.DOD.PVLX.EX.ZS,WDI,ESTRUCTURAL,0.0,0.0,sin_ningun_dato_valido


**Qué comprobar:**
- `len(codigos_incluidos) + df_variables_excluidas.shape[0]` debería ser igual (o muy cercano) al
  total de códigos candidatos en `df_variables` (verificar con
  `df_variables['codigo'].nunique()`); si no cierra, hay códigos que no quedaron ni incluidos ni
  documentados como excluidos — revisar antes de seguir.
- Prestar atención a la distribución por dimensión: si INSTITUCIONAL o ESTRUCTURAL quedan con muy
  pocas variables tras el filtro, el PCA de esa dimensión (cuaderno 15) va a tener poco margen —
  mejor detectarlo ahora que en la sección de PCA.


## 6. Features de trayectoria (media, tendencia, volatilidad residual)

**Objetivo:** para cada país y cada variable incluida, calcular tres estadísticos resumen de su serie
2000-2020: **media**, **tendencia** (pendiente de la regresión lineal valor ~ año) y **volatilidad**
(desviación estándar de los residuos de esa misma regresión — volatilidad neta de tendencia, no la SD
del nivel bruto).

**Justificación metodológica:** calcular la SD directamente sobre el nivel confundiría tendencia con
volatilidad (una serie con tendencia fuerte tiene SD alta aunque sea perfectamente lineal, sin nada de
inestabilidad año a año). Usar los residuos del mismo ajuste que da la tendencia aísla la variabilidad
de corto plazo. Se descarta el coeficiente de variación (SD/media): con variables que pueden tomar
valores cercanos a cero o negativos (varias del bloque monetario-fiscal), el CV se vuelve inestable o
pierde sentido; como de todos modos se va a estandarizar antes del PCA, no aporta nada que el z-score
no resuelva.

Se exige un mínimo de `MIN_ANIOS_TENDENCIA` años con dato válido para calcular tendencia/volatilidad
de forma confiable; con menos años, se calcula la media pero tendencia/volatilidad quedan como NaN
(no se inventa una recta con 2-3 puntos).

**Resultado esperado:** `df_trayectorias_largo` (país, codigo, n_anios, media, tendencia, volatilidad).

**Nota de rendimiento:** este bloque puede tardar uno o dos minutos — es un `groupby().apply()` sobre
potencialmente decenas de miles de combinaciones país-variable. Es normal, no indica error.


In [12]:
MIN_ANIOS_TENDENCIA = 5  # minimo de anios con dato valido para calcular tendencia/volatilidad

def calcular_features_trayectoria(grupo):
    valores = grupo.dropna(subset=["valor"])
    n = len(valores)
    media = valores["valor"].mean() if n > 0 else np.nan

    if n >= MIN_ANIOS_TENDENCIA:
        slope, intercept, r_value, p_value, std_err = stats.linregress(valores["year"], valores["valor"])
        residuos = valores["valor"] - (slope * valores["year"] + intercept)
        volatilidad = residuos.std(ddof=1)
        tendencia = slope
    else:
        tendencia = np.nan
        volatilidad = np.nan

    return pd.Series({"n_anios": n, "media": media, "tendencia": tendencia, "volatilidad": volatilidad})


df_variables_incluidas = df_variables[df_variables["codigo"].isin(codigos_incluidos)].copy()

df_trayectorias_largo = (
    df_variables_incluidas
    .groupby(["ISO-alpha3", "codigo"])
    .apply(calcular_features_trayectoria, include_groups=False)
    .reset_index()
)

print(f"Shape df_trayectorias_largo: {df_trayectorias_largo.shape}")
print(f"Combinaciones pais-variable con tendencia/volatilidad NaN (menos de {MIN_ANIOS_TENDENCIA} anios): "
      f"{df_trayectorias_largo['tendencia'].isna().sum()} de {df_trayectorias_largo.shape[0]}")
df_trayectorias_largo.head()


Shape df_trayectorias_largo: (130980, 6)
Combinaciones pais-variable con tendencia/volatilidad NaN (menos de 5 anios): 11208 de 130980


,ISO-alpha3,codigo,n_anios,media,tendencia,volatilidad
0,AFG,AG.LND.TOTL.K2,21.0,6.522300e+05,0.000000e+00,0.000000e+00
1,AFG,BG.GSR.NFSV.GD.ZS,13.0,1.441609e+01,-1.201048e+00,2.530046e+00
2,AFG,BM.GSR.CMCP.ZS,13.0,4.855776e+01,-8.250127e+00,1.568010e+01
3,AFG,BM.GSR.FCTY.CD,13.0,1.102253e+08,-8.859798e+06,3.155010e+07
4,AFG,BM.GSR.GNFS.CD,13.0,7.302868e+09,2.234023e+08,1.668319e+09


**Qué comprobar:**
- Si el porcentaje de NaN en `tendencia` es muy alto (por ejemplo, más del 20-30% de las
  combinaciones), reconsiderar `MIN_ANIOS_TENDENCIA` — puede ser demasiado exigente dado que muchas
  variables de V-Dem tienen cortes temporales estructurales ya documentados en el cuaderno 07.
- Error común: `include_groups=False` requiere pandas ≥ 2.2. Si tu versión es anterior, sacar ese
  argumento del `.apply()` (el comportamiento por defecto en versiones viejas ya excluye las columnas
  de agrupación del `grupo` que llega a la función).


## 7. Target de inflación: mismas features

**Objetivo:** aplicar el mismo cálculo de media/tendencia/volatilidad al target principal
(`FP.CPI.TOTL.ZG`) y al secundario (`NY.GDP.DEFL.KD.ZG`), para tener la trayectoria de inflación de
cada país lista para cruzar con los grupos del clustering de etapa 1.

**Justificación metodológica:** reutilizar exactamente la misma función asegura que la trayectoria de
inflación es conceptual y metodológicamente comparable con las trayectorias del resto de variables
(mismos supuestos, mismo umbral mínimo de años).

**Resultado esperado:** `df_target_features` (por país: `n_anios, media, tendencia, volatilidad` de
inflación IPC) y `df_target_secundario_features` (ídem para el deflactor).


In [13]:
df_target_features = (
    df_target_inflacion
    .groupby("ISO-alpha3")
    .apply(calcular_features_trayectoria, include_groups=False)
    .reset_index()
)

df_target_secundario_features = (
    df_target_secundario
    .groupby("ISO-alpha3")
    .apply(calcular_features_trayectoria, include_groups=False)
    .reset_index()
)

print(f"Paises con features de inflacion (target principal): {df_target_features.shape[0]}")
print(f"Paises con tendencia/volatilidad valida (target principal): {df_target_features['tendencia'].notna().sum()}")
print(f"Paises con features de deflactor (target secundario): {df_target_secundario_features.shape[0]}")

df_target_features.describe()


Paises con features de inflacion (target principal): 193
Paises con tendencia/volatilidad valida (target principal): 181
Paises con features de deflactor (target secundario): 193


,n_anios,media,tendencia,volatilidad
count,193.000000,184.000000,181.000000,181.000000
mean,19.005181,7.115094,0.104414,6.303463
std,5.242552,11.443013,3.649996,16.133391
min,0.000000,0.087775,-16.827604,0.586045
25%,21.000000,2.027571,-0.282609,1.636190
50%,21.000000,3.866674,-0.112877,2.588923
75%,21.000000,7.209491,-0.033024,4.406627
max,21.000000,76.767205,34.510036,134.880278


**Qué comprobar:**
- `df_target_features['tendencia'].notna().sum()` es, en la práctica, el número máximo de países que
  van a poder entrar al análisis de perfiles de trayectoria inflacionaria — si es sensiblemente menor
  a 193, documentarlo como limitación de cobertura del target.
- Revisar con `describe()` que no haya valores de inflación extremos que sean errores de carga (por
  ejemplo, inflación de miles de % por hiperinflaciones puntuales — son reales, pero conviene
  identificar qué países son antes del PCA para decidir si necesitan tratamiento aparte, ej. winsorizing).


## 8. Pivote a formato ancho (país × variable_estadístico)

**Objetivo:** pasar `df_trayectorias_largo` de formato largo (país, codigo, estadístico) a una única
fila por país con columnas `<codigo>_media`, `<codigo>_tendencia`, `<codigo>_volatilidad` — el formato
que necesita el PCA del cuaderno 15.

**Justificación metodológica:** el PCA y el clustering trabajan sobre una matriz observaciones ×
variables; la unidad de observación pasa a ser el país (n≤193), no el par país-año.

**Resultado esperado:** `df_pais_features`, indexado por `ISO-alpha3`, con hasta 3× el número de
variables incluidas como columnas.


In [14]:
df_media = df_trayectorias_largo.pivot(index="ISO-alpha3", columns="codigo", values="media")
df_media.columns = [f"{c}_media" for c in df_media.columns]

df_tendencia = df_trayectorias_largo.pivot(index="ISO-alpha3", columns="codigo", values="tendencia")
df_tendencia.columns = [f"{c}_tendencia" for c in df_tendencia.columns]

df_volatilidad = df_trayectorias_largo.pivot(index="ISO-alpha3", columns="codigo", values="volatilidad")
df_volatilidad.columns = [f"{c}_volatilidad" for c in df_volatilidad.columns]

df_pais_features = pd.concat([df_media, df_tendencia, df_volatilidad], axis=1)

print(f"Shape df_pais_features: {df_pais_features.shape}")
print(f"Paises (filas): {df_pais_features.shape[0]} de {N_PAISES_UNIVERSO}")
print(f"Features (columnas): {df_pais_features.shape[1]} (de {len(codigos_incluidos)} variables x 3 estadisticos)")
df_pais_features.head()


Shape df_pais_features: (193, 2058)
Paises (filas): 193 de 193
Features (columnas): 2058 (de 686 variables x 3 estadisticos)


AG.LND.TOTL.K2_media  BG.GSR.NFSV.GD.ZS_media  \
ISO-alpha3                                                  
AFG                     652230.0                14.416094   
AGO                    1246700.0                19.370284   
ALB                      27400.0                33.667719   
AND                        470.0                81.447556   
ARE                      71020.0                      NaN   

            BM.GSR.CMCP.ZS_media  BM.GSR.FCTY.CD_media  BM.GSR.GNFS.CD_media  \
ISO-alpha3                                                                     
AFG                    48.557755          1.102253e+08          7.302868e+09   
AGO                    35.735548          6.633857e+09          2.690268e+10   
ALB                    13.514135          2.690858e+08          4.870589e+09   
AND                    41.654165          2.187798e+08          1.879695e+09   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.INSF.ZS_media  BM.GSR.MRCH.CD_media  BM.GSR.NFSV.CD_media  \
ISO-alpha3                                                                     
AFG                     2.040682          6.048209e+09          1.254658e+09   
AGO                     4.559638          1.423333e+10          1.266935e+10   
ALB                     4.229901          3.228758e+09          1.641832e+09   
AND                     4.325387          1.348060e+09          5.316355e+08   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.ROYL.CD_media  BM.GSR.TOTL.CD_media  BM.GSR.TRAN.ZS_media  \
ISO-alpha3                                                                     
AFG                 8.508323e+06          7.413093e+09             70.091566   
AGO                 1.008403e+08          3.353654e+10             21.448274   
ALB                 1.758352e+07          5.139675e+09             15.520530   
AND                 3.736637e+05          2.098475e+09             13.758242   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.TRVL.ZS_media  BM.KLT.DINV.CD.WD_media  \
ISO-alpha3                                                  
AFG                     9.129564             7.591159e+06   
AGO                     2.550764             5.048538e+08   
ALB                    66.826756             7.618535e+07   
AND                    29.053041             1.174413e+08   
ARE                          NaN             8.620458e+09   

            BM.KLT.DINV.WD.GD.ZS_media  BM.TRF.PRVT.CD_media  \
ISO-alpha3                                                     
AFG                           0.041417          3.978885e+08   
AGO                           0.635668          5.592257e+08   
ALB                           0.604012          1.522573e+08   
AND                           3.914739          5.778313e+07   
ARE                           2.591342                   NaN   

            BM.TRF.PWKR.CD.DT_media  BN.CAB.XOKA.CD_media  \
ISO-alpha3                                                  
AFG                    3.534705e+08         -2.888089e+09   
AGO                    8.341721e+08          3.021819e+09   
ALB                    1.262605e+08         -1.005074e+09   
AND                    8.764456e+07          5.085633e+08   
ARE                             NaN                   NaN   

            BN.CAB.XOKA.GD.ZS_media  BN.FIN.TOTL.CD_media  \
ISO-alpha3                                                  
AFG                      -15.123911          5.884412e+08   
AGO                        3.098622          1.750814e+09   
ALB                       -8.867333         -7.373159e+08   
AND                       16.768451          5.374083e+08   
ARE                             NaN                   NaN   

            BN.GSR.FCTY.CD_media  BN.GSR.GNFS.CD_media  BN.GSR.MRCH.CD_media  \
ISO-alpha3                                                                     
AFG     

**Qué comprobar:**
- Filas debería ser ≤193 (solo países con al menos un dato en alguna variable incluida). Si es
  sensiblemente menor a 193, revisar qué países quedaron fuera y por qué (probablemente los mismos ya
  documentados como ausentes de V-Dem/MC en cuadernos previos).
- Columnas debería ser exactamente `3 * len(codigos_incluidos)`.


## 9. Missingness a nivel país tras la agregación

**Objetivo:** cuantificar, en `df_pais_features`, qué porcentaje de celdas está vacío por fila (país)
y por columna (variable-estadístico), como insumo directo para la decisión de imputación que toca en
el cuaderno 15 antes del PCA.

**Justificación metodológica:** el filtro de disponibilidad de la sección 5 se aplicó a nivel
variable (cobertura agregada); acá se mide el missingness remanente a nivel de la matriz final país ×
feature, que es lo que realmente entra al PCA. Los dos niveles no son intercambiables — una variable
puede cumplir el umbral global y aun así faltarle el dato a países puntuales.

**Resultado esperado:** dos tablas de resumen (por país, por variable) y una lista de países con
missingness alto para revisar antes de decidir la estrategia de imputación.


In [15]:
missingness_por_columna = (
    df_pais_features.isna().mean().sort_values(ascending=False) * 100
).round(1)

missingness_por_pais = (
    df_pais_features.isna().mean(axis=1).sort_values(ascending=False) * 100
).round(1)

print("Columnas con mayor % de missingness:")
print(missingness_por_columna.head(10))

print("\nPaises con mayor % de missingness:")
print(missingness_por_pais.head(15))

UMBRAL_MISSINGNESS_PAIS = 40  # % de features vacias por pais, solo para señalar casos a revisar
paises_alta_missingness = missingness_por_pais[missingness_por_pais > UMBRAL_MISSINGNESS_PAIS]
print(f"\nPaises con mas de {UMBRAL_MISSINGNESS_PAIS}% de features vacias: {len(paises_alta_missingness)}")


Columnas con mayor % de missingness:
GB.XPD.RSDV.GD.ZS_volatilidad    43.0
GB.XPD.RSDV.GD.ZS_tendencia      43.0
IP.PAT.RESD_tendencia            35.8
IP.PAT.RESD_volatilidad          35.8
BX.GSR.ROYL.CD_volatilidad       31.6
BX.GSR.ROYL.CD_tendencia         31.6
GC.TAX.IMPT.ZS_tendencia         30.6
GC.TAX.IMPT.ZS_volatilidad       30.6
GC.TAX.IMPT.CN_volatilidad       30.6
BX.PEF.TOTL.CD.WD_tendencia      30.6
dtype: float64

Paises con mayor % de missingness:
ISO-alpha3
MCO    70.9
LIE    67.4
AND    60.3
PRK    55.2
TUV    50.1
NRU    47.2
SMR    44.4
FSM    41.7
GRD    41.1
DMA    39.1
ATG    37.2
KNA    36.4
TKM    33.9
PLW    33.8
MHL    33.0
dtype: float64

Paises con mas de 40% de features vacias: 9


**Qué comprobar:**
- Esta sección es diagnóstica, no decide nada todavía — la estrategia de imputación (eliminación,
  imputación simple, KNN, MICE) se define y justifica en el cuaderno 15, comparando alternativas como
  indican los principios metodológicos del proyecto.
- Si `paises_alta_missingness` incluye países muy pequeños o territorios con baja cobertura
  estadística internacional (patrón ya visto en V-Dem/MC en cuadernos anteriores), es consistente con
  lo ya documentado — no es un error nuevo de este cuaderno.


## 10. Exportación

**Objetivo:** guardar los artefactos de este cuaderno para que el cuaderno 15 (PCA por dimensión)
pueda partir directamente de ellos sin recalcular nada.

**Justificación metodológica:** separar selección de variables + construcción de features (este
cuaderno) del PCA (próximo cuaderno) mantiene cada cuaderno con un alcance acotado, siguiendo el
diseño modular ya usado en el resto del proyecto.

**Resultado esperado:** `14_pais_features.xlsx` con las hojas `pais_features`, `target_inflacion`,
`target_secundario`, `disponibilidad_variables`, `variables_excluidas`. Celdas de exportación
comentadas por defecto.


In [16]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "14_pais_features.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_pais_features.to_excel(writer, sheet_name="pais_features")
    df_target_features.to_excel(writer, sheet_name="target_inflacion", index=False)
    df_target_secundario_features.to_excel(writer, sheet_name="target_secundario", index=False)
    df_disponibilidad_variables.to_excel(writer, sheet_name="disponibilidad_variables", index=False)
    df_variables_excluidas.to_excel(writer, sheet_name="variables_excluidas", index=False)
print(f"Exportado: {out_path}")


Exportado: data\processed\14_pais_features.xlsx
